In [26]:
import os
from dotenv import load_dotenv
import json
from pydantic import BaseModel, Field

load_dotenv("../.env")   

True

In [27]:
from unstructured.partition.pdf import partition_pdf
from collections import defaultdict

# Parse the PDF
elements = partition_pdf("../data/3.pdf")

# Create a dictionary to hold elements grouped by page number
pages = defaultdict(list)

for element in elements:
    # Ensure the element has a page number in its metadata
    if element.metadata.page_number:
        page_num = element.metadata.page_number
        pages[page_num].append(element.text)

# Sort the pages to ensure they are in order (1, 2, 3...)
# Then join every page's text into one final string
full_document_text = "\n\n".join(["\n".join(pages[i]) for i in sorted(pages.keys())])

print(full_document_text)


Cannot set non-stroke color: /'Pat9' is an invalid float value


View IFT /PQ / REOI / RFP / PPS Notice Details
Ministry :
Organization :
Ministry of Local Government, Rural Development and Co- operatives Dhaka South City Corporation
Procuring Entity Code : Procurement Nature : Goods Event Type : Invitation Reference No. : App ID :
TENDER
46.207.026.09.25.040.2026- 098
229719
Division :
Procuring Entity Name :
Procuring Entity District : Procurement Type : Invitation for :
Tender/Proposal ID :
Local Government Division
Office of the Executive Engineer Mechanical (DSCC) Dhaka
NCT
Tender - Single Lot
1318452
Key Information and Funding Information :
Procurement Method : Open Tendering Method
Budget Type :
Own Fund
Source of Funds :
(OTM) Own Fund
Particular Information :
Project Code : Tender/Proposal Package No. and Description :
Category :
Not applicable
Project Name :
Not applicable
egpdsccmech-038/2026-27 Engagement of yearly contractor for suppling Stone ships, Sylhet sand, Bitumen & Labour for repair and maintenance of defected bitumenous roads 

In [28]:
# --------------------------------------------------
# 1. Structured output schema
# --------------------------------------------------

class ProcurementDocument(BaseModel):
    procurement_nature: str = Field(
        description="The nature/type of the procurement."
    )

    procurement_method: str = Field(
        description="The procurement method used."
    )

    title: str = Field(
        description="The title of the procurement/tender."
    )

    category: str = Field(
        description="The category of the procurement."
    )

    eligibility_of_tenderer: str = Field(
        description="Eligibility requirements for the tenderer."
    )

    description: str = Field(
        description="Brief description of the works."
    )


In [29]:
# System prompt for the agent
SYSTEM_PROMPT = """
You are a Document Intelligence Agent.

Your task is to extract structured information from a procurement document.

Rules:
- Extract information only from the document.
- Do not invent or hallucinate information.
- If a requested field is not present, return an empty string.
- Preserve the meaning of the original document.
- Extract the complete relevant information for each field.
"""

print("System prompt created")
print(f"Total length: {len(SYSTEM_PROMPT)} characters")

System prompt created
Total length: 387 characters


In [30]:
# Initialize the agent
from langchain_groq import ChatGroq 
groq_key = os.getenv("GROQ_API_KEY")

# LLM for the agent
agent_llm = ChatGroq(
    # model="llama-3.3-70b-versatile",
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=1,
    stop=None
)

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent

agent = create_agent(
    model=agent_llm,
    system_prompt=SYSTEM_PROMPT,
    response_format=ProcurementDocument
)

In [31]:
# --------------------------------------------------
# Invoke agent
# --------------------------------------------------

response = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": f"""
Extract the required information from the following procurement document:

{full_document_text}
"""
        }
    ]
})


# --------------------------------------------------
# Get structured result
# --------------------------------------------------

structured_result = response["structured_response"]

print(structured_result)

procurement_nature='Goods' procurement_method='Open Tendering Method' title='Engagement of yearly contractor for supplying Stone ships, Sylhet sand, Bitumen & Labour for repair and maintenance of defected bitumenous roads of different zones under DSCC through Asphalt Plant of mechanical division (Part-1)' category='Not applicable' eligibility_of_tenderer='The required number of similar contracts like Stone & Bitumen Supply/Road Construction work completed shall be 2(two) Contract at least Tk. 12.00 crore over a period of last 5 (Five) years. The minimum amount of free funds (liquid assets) and/or credit facilities net of other contractual commitments of the successful tenderer shall be 10.00 Crore. The Tenderer must submit the attested photocopy of their up to date trade licence, income tax, VAT and registration certificate.' description='Engagement of yearly contractor for supplying Stone ships, Sylhet sand, Bitumen & Labour for repair and maintenance of defected bitumenous roads of d

In [34]:
print(structured_result.title)
print(structured_result.procurement_method)
print(structured_result.category)
print(structured_result.eligibility_of_tenderer)
print(structured_result.description)

Engagement of yearly contractor for supplying Stone ships, Sylhet sand, Bitumen & Labour for repair and maintenance of defected bitumenous roads of different zones under DSCC through Asphalt Plant of mechanical division (Part-1)
Open Tendering Method
Not applicable
The required number of similar contracts like Stone & Bitumen Supply/Road Construction work completed shall be 2(two) Contract at least Tk. 12.00 crore over a period of last 5 (Five) years. The minimum amount of free funds (liquid assets) and/or credit facilities net of other contractual commitments of the successful tenderer shall be 10.00 Crore. The Tenderer must submit the attested photocopy of their up to date trade licence, income tax, VAT and registration certificate.
Engagement of yearly contractor for supplying Stone ships, Sylhet sand, Bitumen & Labour for repair and maintenance of defected bitumenous roads of different zones under DSCC through Asphalt Plant of mechanical division (Part-1)


In [37]:
from sentence_transformers import SentenceTransformer

# 1. Load the pre-trained all-MiniLM-L6-v2 model
# This model maps sentences & paragraphs to a 384-dimensional dense vector space
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Generate the embeddings
print("Generating embeddings...")
embedding = model.encode(structured_result.description, show_progress_bar=True)

# 3. Review the output
print(f"Embedding Shape: {embedding.shape}")  # Will be (384,)
print(f"Sample Vector Values (First 5 dimensions): {embedding[:5]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding Shape: (384,)
Sample Vector Values (First 5 dimensions): [-0.08716753 -0.01314992  0.12647891  0.01782808 -0.08848882]


In [ ]:
# from supabase import create_client, Client
# from sentence_transformers import SentenceTransformer

# # 1. Supabase Credentials 
# SUPABASE_URL = os.environ["SUPABASE_URL"]
# SUPABASE_KEY = os.environ["SUPABASE_KEY"]

# # Initialize the Supabase client
# supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# # CRITICAL STEP: Supabase expects a standard Python list, not a NumPy array!
# # Convert it using .tolist() before inserting.
# embedding_list = embedding.tolist()

# print("Inserting into Supabase...")

# response = supabase.table("tenders").insert({
#     "title": title,
#     "embedding": embedding_list
# }).execute()

# print("Success!")
# print(f"Inserted ID: {response.data[0]['tender_id']}")